# Great Expectations for Data Quality in Snowflake

This notebook demonstrates how to use **Great Expectations (GX)** for data quality validation with Snowflake.

## What is Great Expectations?
- Open-source Python library for data validation and documentation
- Define "expectations" (tests) about your data
- Validate data against expectations and get detailed reports
- Create data documentation automatically

## What We'll Cover
1. Installing and configuring Great Expectations
2. Connecting to Snowflake
3. Creating an Expectation Suite
4. Validating data against expectations
5. Viewing validation results
6. Comparison with Snowflake's native DMFs

In [1]:
import sys
!{sys.executable} -m pip install great_expectations snowflake-sqlalchemy -q

print("Great Expectations installed successfully!")

In [3]:
pip install great_expectations

In [5]:
import great_expectations
print(great_expectations.__file__)

In [7]:
import great_expectations as gx
import pandas as pd
from snowflake.snowpark.context import get_active_session

print(f"Great Expectations version: {gx.__version__}")

In [8]:
session = get_active_session()
print(f"Connected to Snowflake")
print(f"Current database: {session.get_current_database()}")
print(f"Current schema: {session.get_current_schema()}")

---
## Step 1: Create Sample Data for Testing

Let's create a sample dataset to validate with Great Expectations.

In [9]:
%%sql -r sample_orders
SELECT 
    ROW_NUMBER() OVER (ORDER BY RANDOM()) AS order_id,
    CASE WHEN RANDOM() > 0.05 
         THEN 'customer_' || UNIFORM(1, 1000, RANDOM())::STRING 
         ELSE NULL END AS customer_id,
    DATEADD(day, -UNIFORM(1, 365, RANDOM()), CURRENT_DATE()) AS order_date,
    ROUND(UNIFORM(10, 1000, RANDOM()) + RANDOM() * 100, 2) AS order_amount,
    ARRAY_CONSTRUCT('pending', 'completed', 'shipped', 'cancelled', 'invalid_status')[UNIFORM(0, 4, RANDOM())]::STRING AS status,
    UNIFORM(1, 100, RANDOM()) AS quantity,
    CASE WHEN RANDOM() > 0.1 THEN 'test@example.com' ELSE 'invalid-email' END AS email
FROM TABLE(GENERATOR(ROWCOUNT => 1000))

In [10]:
df = sample_orders
print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nSample data:")
df.head(10)

---
## Step 2: Initialize Great Expectations Context

Create an ephemeral (in-memory) Data Context for validation.

In [11]:
context = gx.get_context()
print("Great Expectations context created successfully!")
print(f"Context type: {type(context).__name__}")

---
## Step 3: Add Data Source and Create Batch

Add our pandas DataFrame as a data source for validation.

In [13]:
datasource = context.data_sources.add_pandas(name="orders_datasource")

data_asset = datasource.add_dataframe_asset(name="orders_data")

batch_definition = data_asset.add_batch_definition_whole_dataframe("orders_batch")

print("Data source and batch created successfully!")

---
## Step 4: Create Expectation Suite

Define a set of expectations (data quality rules) for our orders data.

In [14]:
suite_name = "orders_quality_suite"
suite = gx.ExpectationSuite(name=suite_name)
suite = context.suites.add(suite)

print(f"Created expectation suite: {suite_name}")

In [15]:
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})
validator = context.get_validator(
    batch=batch,
    expectation_suite=suite
)

print(f"Validator created successfully!")

---
## Step 5: Define Individual Expectations

Now let's add various data quality expectations to our suite.

In [16]:
result = validator.expect_table_row_count_to_be_between(
    min_value=500,
    max_value=2000
)

print(f"Row count expectation: {'PASSED' if result.success else 'FAILED'}")
print(f"Actual row count: {result.result.get('observed_value')}")

In [17]:
required_columns = ['ORDER_ID', 'CUSTOMER_ID', 'ORDER_DATE', 'ORDER_AMOUNT', 'STATUS']

for col in required_columns:
    result = validator.expect_column_to_exist(column=col)
    status = 'PASSED' if result.success else 'FAILED'
    print(f"Column '{col}' exists: {status}")

In [18]:
result = validator.expect_column_values_to_be_unique(column="ORDER_ID")

print(f"ORDER_ID uniqueness: {'PASSED' if result.success else 'FAILED'}")
if not result.success:
    print(f"Duplicate count: {result.result.get('unexpected_count')}")

In [19]:
result = validator.expect_column_values_to_not_be_null(
    column="CUSTOMER_ID",
    mostly=0.95  # Allow up to 5% nulls
)

print(f"CUSTOMER_ID not null (95%): {'PASSED' if result.success else 'FAILED'}")
print(f"Null percentage: {(1 - result.result.get('unexpected_percent', 0)/100) * 100:.2f}%")

In [20]:
result = validator.expect_column_values_to_be_between(
    column="ORDER_AMOUNT",
    min_value=0,
    max_value=5000
)

print(f"ORDER_AMOUNT range [0, 5000]: {'PASSED' if result.success else 'FAILED'}")
print(f"Min observed: {result.result.get('observed_value', {}).get('min', 'N/A')}")
print(f"Max observed: {result.result.get('observed_value', {}).get('max', 'N/A')}")

In [21]:
valid_statuses = ['pending', 'completed', 'shipped', 'cancelled']

result = validator.expect_column_values_to_be_in_set(
    column="STATUS",
    value_set=valid_statuses
)

print(f"STATUS in valid set: {'PASSED' if result.success else 'FAILED'}")
if not result.success:
    print(f"Invalid values found: {result.result.get('partial_unexpected_list', [])}")

In [22]:
email_pattern = r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$'

result = validator.expect_column_values_to_match_regex(
    column="EMAIL",
    regex=email_pattern,
    mostly=0.8  # At least 80% should match
)

print(f"EMAIL format valid (80%): {'PASSED' if result.success else 'FAILED'}")
match_pct = 100 - (result.result.get('unexpected_percent', 0))
print(f"Valid email percentage: {match_pct:.1f}%")

In [24]:
from datetime import datetime, timedelta

min_date = (datetime.now() - timedelta(days=400)).strftime('%Y-%m-%d')
max_date = datetime.now().strftime('%Y-%m-%d')

# Convert ORDER_DATE to string for comparison (already in YYYY-MM-DD format)
result = validator.expect_column_values_to_be_between(
    column="ORDER_DATE",
    min_value=min_date,
    max_value=max_date
)

print(f"ORDER_DATE in valid range: {'PASSED' if result.success else 'FAILED'}")
print(f"Expected range: {min_date} to {max_date}")

In [25]:
suite = validator.expectation_suite

# Delete existing suite if it exists, then add
try:
    context.suites.delete(suite_name)
except Exception:
    pass
context.suites.add(suite)

print(f"Expectation suite '{suite_name}' saved!")
print(f"Total expectations: {len(suite.expectations)}")

---
## Step 6: Run Full Validation

Execute all expectations and get a comprehensive validation report.

In [ ]:
validation_definition = gx.ValidationDefinition(
    name="orders_validation",
    data=batch_definition,
    suite=suite
)
validation_definition = context.validation_definitions.add(validation_definition)

validation_result = batch.validate(suite)

print(f"Validation completed!")
print(f"Overall success: {validation_result.success}")

In [ ]:
results = validation_result.results

print("=" * 60)
print("VALIDATION RESULTS SUMMARY")
print("=" * 60)

passed = 0
failed = 0

for r in results:
    exp_type = r.expectation_config.type
    success = r.success
    
    if success:
        passed += 1
        status = "PASS"
    else:
        failed += 1
        status = "FAIL"
    
    column = r.expectation_config.kwargs.get('column', 'N/A')
    
    print(f"[{status}] {exp_type}")
    if column != 'N/A':
        print(f"       Column: {column}")

print("=" * 60)
print(f"Total: {passed + failed} | Passed: {passed} | Failed: {failed}")
print(f"Pass Rate: {passed / (passed + failed) * 100:.1f}%")

---
## Comparison: Great Expectations vs Snowflake DMFs

| Feature | Great Expectations | Snowflake DMFs |
|---------|-------------------|----------------|
| **Setup** | Requires Python package | Native SQL |
| **Execution** | In notebook/Python | Scheduled on Snowflake |
| **Expectations** | 300+ built-in | System + Custom DMFs |
| **Results** | JSON/HTML reports | Event tables, views |
| **Best For** | Ad-hoc validation, CI/CD | Production monitoring |
| **Cost** | Compute during execution | Cloud services billing |

---
## Bonus: Equivalent Snowflake DMF Approach

Here's how you would implement similar checks using Snowflake's native Data Metric Functions.

In [ ]:
%%sql -r dmf_example
-- Example: Setting up native Snowflake DMFs (requires table, shown for reference)
-- These would be run on an actual table, not a query result

SELECT 
    'NULL_COUNT' AS dmf_name,
    'Counts null values in a column' AS description,
    'ALTER TABLE t ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.NULL_COUNT ON (col) EXPECTATION my_exp (VALUE < 10)' AS example_sql
UNION ALL
SELECT 'ROW_COUNT', 'Counts total rows', 'ALTER TABLE t ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.ROW_COUNT ON () EXPECTATION (VALUE > 0)'
UNION ALL  
SELECT 'DUPLICATE_COUNT', 'Counts duplicate values', 'ALTER TABLE t ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.DUPLICATE_COUNT ON (col)'
UNION ALL
SELECT 'FRESHNESS', 'Measures data freshness', 'ALTER TABLE t ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.FRESHNESS ON (timestamp_col)'

---
## Custom Expectations Example

Great Expectations allows creating custom expectations for domain-specific validations.

In [ ]:
high_value_orders = df[df['ORDER_AMOUNT'] > 500]
high_value_with_customer = high_value_orders[high_value_orders['CUSTOMER_ID'].notna()]

rule_pass = len(high_value_with_customer) / len(high_value_orders) >= 0.99 if len(high_value_orders) > 0 else True

print("Custom Business Rule: High-value orders (>$500) must have customer_id")
print(f"High-value orders: {len(high_value_orders)}")
print(f"With customer_id: {len(high_value_with_customer)}")
print(f"Compliance rate: {len(high_value_with_customer) / len(high_value_orders) * 100:.1f}%")
print(f"Rule status: {'PASSED' if rule_pass else 'FAILED'}")

---
## Summary

This notebook demonstrated:

### Great Expectations Capabilities
- **Row count validation** - `expect_table_row_count_to_be_between`
- **Column existence** - `expect_column_to_exist`
- **Uniqueness checks** - `expect_column_values_to_be_unique`
- **Null checks** - `expect_column_values_to_not_be_null`
- **Range validation** - `expect_column_values_to_be_between`
- **Enum validation** - `expect_column_values_to_be_in_set`
- **Regex patterns** - `expect_column_values_to_match_regex`
